# 🧬 Parallel DNA Sequence Alignment Using CUDA
## BCS702 — Parallel Computing | CCA 1 | Semester VII | Section D

**Algorithm:** Needleman-Wunsch Global Sequence Alignment  
**Parallelism:** CUDA Wavefront / Anti-Diagonal Kernel  
**GPU:** NVIDIA T4 (Google Colab free tier)

---
### ⚠️ IMPORTANT: Before running
1. Go to **Runtime → Change runtime type**
2. Set **Hardware accelerator = GPU** (T4)
3. Click **Save**, then **Runtime → Run all**

In [ ]:
# ── CELL 1: Verify GPU is available ────────────────────────
!nvidia-smi
!nvcc --version

In [ ]:
# ── CELL 2: Write the CUDA source file ─────────────────────
cuda_code = r"""
// =============================================================================
// DNA SEQUENCE ALIGNMENT USING CUDA
// =============================================================================
// Course  : Parallel Computing (BCS702), Semester VII, Section D
// Project : Parallel DNA Sequence Alignment Using CUDA (CCA 1)
// Algorithm: Needleman-Wunsch Global Sequence Alignment
//
// GPU Kernel uses WAVEFRONT / ANTI-DIAGONAL PARALLELISM:
//   cells with same anti-diagonal index (i+j = constant) are independent
//   and computed in parallel. One kernel launch per diagonal.
//
// Scoring: Match=+2, Mismatch=-1, Gap=-2
// =============================================================================

#include <cuda_runtime.h>
#include <iostream>
#include <fstream>
#include <string>
#include <vector>
#include <algorithm>
#include <chrono>
#include <random>
#include <iomanip>
#include <cstdlib>

using namespace std;

// Scoring constants
const int MATCH    =  2;
const int MISMATCH = -1;
const int GAP      = -2;

// CUDA error check macro
#define CUDA_CHECK(call) do { \
    cudaError_t err = (call); \
    if (err != cudaSuccess) { \
        fprintf(stderr, "[CUDA ERROR] %s at %s:%d\n", \
                cudaGetErrorString(err), __FILE__, __LINE__); \
        exit(EXIT_FAILURE); \
    } \
} while(0)

// ── Generate random DNA string ─────────────────────────────
static string randomDNA(int n, unsigned seed = 42) {
    static const char bases[] = "ACGT";
    mt19937 rng(seed);
    uniform_int_distribution<int> dist(0, 3);
    string seq(n, ' ');
    for (int i = 0; i < n; ++i) seq[i] = bases[dist(rng)];
    return seq;
}

// =============================================================================
// CPU IMPLEMENTATION — Sequential Needleman-Wunsch
// Fills the DP matrix row by row (no parallelism)
// =============================================================================
int cpuNeedlemanWunsch(const string& seq1, const string& seq2) {
    int m = (int)seq1.size();
    int n = (int)seq2.size();
    vector<int> dp((m+1)*(n+1), 0);

    for (int j = 0; j <= n; ++j) dp[0*(n+1)+j] = j * GAP;
    for (int i = 0; i <= m; ++i) dp[i*(n+1)+0] = i * GAP;

    for (int i = 1; i <= m; ++i) {
        for (int j = 1; j <= n; ++j) {
            int d = dp[(i-1)*(n+1)+(j-1)] + (seq1[i-1]==seq2[j-1] ? MATCH : MISMATCH);
            int u = dp[(i-1)*(n+1)+j] + GAP;
            int l = dp[i*(n+1)+(j-1)] + GAP;
            dp[i*(n+1)+j] = max({d, u, l});
        }
    }
    return dp[m*(n+1)+n];
}

// =============================================================================
// CUDA KERNEL — nwKernel
// Each thread computes ONE cell on anti-diagonal 'diag'.
// Thread k → row i = i_start + k,  col j = diag - i
// =============================================================================
__global__
void nwKernel(int* d_dp, const char* d_seq1, const char* d_seq2,
              int m, int n, int diag,
              int matchScore, int mismatchScore, int gapScore)
{
    int k = blockIdx.x * blockDim.x + threadIdx.x;

    int i_start  = max(1, diag - n);
    int i_end    = min(m, diag - 1);
    int numCells = i_end - i_start + 1;

    if (k >= numCells) return;

    int i = i_start + k;
    int j = diag - i;

    if (i < 1 || i > m || j < 1 || j > n) return;

    int stride = n + 1;
    int d = d_dp[(i-1)*stride+(j-1)] + (d_seq1[i-1]==d_seq2[j-1] ? matchScore : mismatchScore);
    int u = d_dp[(i-1)*stride+j] + gapScore;
    int l = d_dp[i*stride+(j-1)] + gapScore;
    d_dp[i*stride+j] = max(d, max(u, l));
}

// =============================================================================
// GPU ORCHESTRATOR — cudaNeedlemanWunsch
// Manages: cudaMalloc, cudaMemcpy H->D, kernel launches, cudaMemcpy D->H
// Timing via CUDA Events (most accurate)
// =============================================================================
int cudaNeedlemanWunsch(const string& seq1, const string& seq2,
                        float& kernelMs, float& totalMs)
{
    int m = (int)seq1.size();
    int n = (int)seq2.size();
    int dpSize = (m+1)*(n+1);

    // Initialize DP on host
    vector<int> h_dp(dpSize, 0);
    for (int j = 0; j <= n; ++j) h_dp[j]          = j * GAP;
    for (int i = 0; i <= m; ++i) h_dp[i*(n+1)]    = i * GAP;

    // Device pointers
    int  *d_dp   = nullptr;
    char *d_seq1 = nullptr, *d_seq2 = nullptr;

    // CUDA Events
    cudaEvent_t t0, t1, k0, k1;
    CUDA_CHECK(cudaEventCreate(&t0)); CUDA_CHECK(cudaEventCreate(&t1));
    CUDA_CHECK(cudaEventCreate(&k0)); CUDA_CHECK(cudaEventCreate(&k1));

    // Total time start (includes mem transfers)
    CUDA_CHECK(cudaEventRecord(t0));

    // Allocate device memory
    CUDA_CHECK(cudaMalloc((void**)&d_dp,   dpSize * sizeof(int)));
    CUDA_CHECK(cudaMalloc((void**)&d_seq1, m * sizeof(char)));
    CUDA_CHECK(cudaMalloc((void**)&d_seq2, n * sizeof(char)));

    // Copy Host → Device
    CUDA_CHECK(cudaMemcpy(d_dp,   h_dp.data(),  dpSize*sizeof(int),  cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_seq1, seq1.c_str(), m*sizeof(char),      cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_seq2, seq2.c_str(), n*sizeof(char),      cudaMemcpyHostToDevice));

    const int THREADS = 256;

    // Kernel time start
    CUDA_CHECK(cudaEventRecord(k0));

    // ── WAVEFRONT LOOP ─────────────────────────────────────────
    // For each anti-diagonal d = 2 .. m+n:
    //   launch one kernel — all cells on diagonal d computed in parallel
    //   cudaDeviceSynchronize() = wavefront barrier (like __syncthreads across blocks)
    for (int diag = 2; diag <= m + n; ++diag) {
        int i_start  = max(1, diag - n);
        int i_end    = min(m, diag - 1);
        int numCells = i_end - i_start + 1;
        if (numCells <= 0) continue;
        int grid = (numCells + THREADS - 1) / THREADS;
        nwKernel<<<grid, THREADS>>>(d_dp, d_seq1, d_seq2, m, n, diag, MATCH, MISMATCH, GAP);
        CUDA_CHECK(cudaDeviceSynchronize());
    }

    // Kernel time stop
    CUDA_CHECK(cudaEventRecord(k1));
    CUDA_CHECK(cudaEventSynchronize(k1));
    CUDA_CHECK(cudaEventElapsedTime(&kernelMs, k0, k1));

    // Copy Device → Host
    CUDA_CHECK(cudaMemcpy(h_dp.data(), d_dp, dpSize*sizeof(int), cudaMemcpyDeviceToHost));

    // Total time stop
    CUDA_CHECK(cudaEventRecord(t1));
    CUDA_CHECK(cudaEventSynchronize(t1));
    CUDA_CHECK(cudaEventElapsedTime(&totalMs, t0, t1));

    int score = h_dp[m*(n+1)+n];

    CUDA_CHECK(cudaFree(d_dp));
    CUDA_CHECK(cudaFree(d_seq1));
    CUDA_CHECK(cudaFree(d_seq2));
    CUDA_CHECK(cudaEventDestroy(t0)); CUDA_CHECK(cudaEventDestroy(t1));
    CUDA_CHECK(cudaEventDestroy(k0)); CUDA_CHECK(cudaEventDestroy(k1));

    return score;
}

// =============================================================================
// CSV EXPORT
// =============================================================================
void appendCSV(const string& path, int seqLen,
               double cpuMs, float kernelMs, float totalMs,
               int cpuScore, int gpuScore)
{
    bool exists = false;
    { ifstream f(path); exists = f.good(); }
    ofstream csv(path, ios::app);
    if (!exists)
        csv << "SeqLen,CPU_ms,GPUKernel_ms,GPUTotal_ms,SpeedupKernel,SpeedupTotal,CPU_Score,GPU_Score,Validation\n";
    double sk = cpuMs / kernelMs;
    double st = cpuMs / totalMs;
    csv << fixed << setprecision(4)
        << seqLen   << ","
        << cpuMs    << ","
        << kernelMs << ","
        << totalMs  << ","
        << sk       << ","
        << st       << ","
        << cpuScore << ","
        << gpuScore << ","
        << (cpuScore==gpuScore ? "PASS" : "FAIL") << "\n";
}

// =============================================================================
// MAIN — Validation + Benchmark
// =============================================================================
int main() {
    cout << "\n================================================\n";
    cout << " DNA Sequence Alignment Using CUDA — BCS702 CCA1\n";
    cout << "================================================\n";

    // Print GPU info
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    cout << "\n[GPU] " << prop.name << "\n";
    cout << "  Compute Capability : " << prop.major << "." << prop.minor << "\n";
    cout << "  Global Memory      : " << prop.totalGlobalMem/1024/1024 << " MB\n";
    cout << "  Multiprocessors    : " << prop.multiProcessorCount << "\n";
    cout << "  Max Threads/Block  : " << prop.maxThreadsPerBlock << "\n\n";

    // ── VALIDATION TEST ────────────────────────────────────────
    cout << "================================================\n";
    cout << " VALIDATION TEST — ACGT vs ACT\n";
    cout << "================================================\n";
    {
        string s1 = "ACGT", s2 = "ACT";
        int cpu = cpuNeedlemanWunsch(s1, s2);
        float km=0, tm=0;
        int gpu = cudaNeedlemanWunsch(s1, s2, km, tm);
        cout << "  seq1        : " << s1 << "\n";
        cout << "  seq2        : " << s2 << "\n";
        cout << "  CPU Score   : " << cpu << "\n";
        cout << "  GPU Score   : " << gpu << "\n";
        cout << "  Validation  : " << (cpu==gpu ? "PASS ✓" : "FAIL ✗") << "\n\n";
    }

    // ── FULL BENCHMARK ─────────────────────────────────────────
    cout << "================================================\n";
    cout << " BENCHMARK — 500 to 8000 bp (3 trials each)\n";
    cout << "================================================\n\n";

    vector<int> sizes = {500, 1000, 2000, 4000, 8000};
    const int TRIALS = 3;
    const string CSV = "results.csv";

    cout << fixed << setprecision(4);
    cout << setw(8)  << "Size"
         << setw(14) << "CPU (ms)"
         << setw(16) << "GPU Kernel(ms)"
         << setw(14) << "GPU Tot(ms)"
         << setw(12) << "Speedup-K"
         << setw(12) << "Speedup-T"
         << setw(10) << "Valid" << "\n";
    cout << string(90, '-') << "\n";

    for (int sz : sizes) {
        double totalCpu = 0;
        float  totalKernel = 0, totalGPU = 0;
        int    lastCpu = 0, lastGpu = 0;
        bool   allPass = true;

        for (int t = 0; t < TRIALS; ++t) {
            string s1 = randomDNA(sz, 100+t);
            string s2 = randomDNA(sz, 200+t);

            // CPU timing
            auto t0 = chrono::high_resolution_clock::now();
            lastCpu  = cpuNeedlemanWunsch(s1, s2);
            auto t1 = chrono::high_resolution_clock::now();
            totalCpu += chrono::duration<double,milli>(t1-t0).count();

            // GPU timing
            float km=0, gm=0;
            lastGpu = cudaNeedlemanWunsch(s1, s2, km, gm);
            totalKernel += km;
            totalGPU    += gm;

            if (lastCpu != lastGpu) allPass = false;
        }

        double avgCpu    = totalCpu    / TRIALS;
        float  avgKernel = totalKernel / TRIALS;
        float  avgTotal  = totalGPU    / TRIALS;
        double sk = avgCpu / avgKernel;
        double st = avgCpu / avgTotal;

        cout << setw(8)  << sz
             << setw(14) << avgCpu
             << setw(16) << avgKernel
             << setw(14) << avgTotal
             << setw(12) << sk
             << setw(12) << st
             << setw(10) << (allPass ? "PASS" : "FAIL") << "\n";

        appendCSV(CSV, sz, avgCpu, avgKernel, avgTotal, lastCpu, lastGpu);
    }

    cout << string(90, '-') << "\n";
    cout << "\n[Done] Results saved to " << CSV << "\n";
    return 0;
}
"""

with open('main.cu', 'w') as f:
    f.write(cuda_code)

print('✅ main.cu written successfully')
print(f'   Lines: {len(cuda_code.splitlines())}')

In [ ]:
# ── CELL 3: Compile with nvcc ───────────────────────────────
# -O2       = optimize
# -arch=sm_75 = Turing (T4). Colab auto-detects, we use sm_75 as safe default

!nvcc -O2 -arch=sm_75 -o dna_align main.cu && echo '\n✅ Compilation SUCCESSFUL' || echo '\n❌ Compilation FAILED'

In [ ]:
# ── CELL 4: Run Validation + Full Benchmark ─────────────────
!./dna_align

In [ ]:
# ── CELL 5: Read CSV results & plot graphs ──────────────────
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

df = pd.read_csv('results.csv')
print('\n=== CUDA Benchmark Results ===')
print(df[['SeqLen','CPU_ms','GPUKernel_ms','GPUTotal_ms','SpeedupKernel','SpeedupTotal','Validation']].to_string(index=False))

# ── Plot settings ────────────────────────────────────────────
BG   = '#0d1117'
BG2  = '#161b22'
GRID = '#21262d'

def style(ax, fig):
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG2)
    ax.tick_params(colors='#8b949e', labelsize=11)
    ax.xaxis.label.set_color('#c9d1d9')
    ax.yaxis.label.set_color('#c9d1d9')
    ax.title.set_color('#f0f6fc')
    for s in ax.spines.values(): s.set_edgecolor('#30363d')
    ax.grid(True, color=GRID, linewidth=0.8, linestyle='--', alpha=0.8)

# ── Graph 1: Execution Time ───────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
style(ax, fig)

ax.plot(df['SeqLen'], df['CPU_ms'],       'o-', color='#fc8181', lw=2.5, ms=8, label='CPU Sequential')
ax.plot(df['SeqLen'], df['GPUKernel_ms'], 's-', color='#48bb78', lw=2.5, ms=8, label='CUDA GPU Kernel')
ax.plot(df['SeqLen'], df['GPUTotal_ms'],  '^-', color='#63b3ed', lw=2.5, ms=8, label='CUDA GPU Total (incl. H↔D transfer)')
ax.fill_between(df['SeqLen'], df['CPU_ms'], df['GPUKernel_ms'], alpha=0.12, color='#fc8181')

ax.set_xlabel('Sequence Length (bp)', fontsize=13)
ax.set_ylabel('Execution Time (ms)', fontsize=13)
ax.set_title('CPU Sequential vs CUDA GPU Execution Time\nNeedleman-Wunsch DNA Alignment | BCS702 CCA-1', fontsize=14, fontweight='bold', pad=14)
ax.legend(fontsize=11, facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('cuda_time_comparison.png', dpi=150, facecolor=BG)
plt.show()
print('✅ Saved: cuda_time_comparison.png')

# ── Graph 2: Speedup ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
style(ax, fig)

ax.plot(df['SeqLen'], df['SpeedupKernel'], 'o-', color='#9f7aea', lw=2.5, ms=8, label='Speedup (Kernel only)')
ax.plot(df['SeqLen'], df['SpeedupTotal'],  's-', color='#f6ad55', lw=2.5, ms=8, label='Speedup (Total incl. mem transfers)')
ax.axhline(1.0, color='#8b949e', lw=1.2, ls='--', alpha=0.5, label='Baseline (1×)')
ax.fill_between(df['SeqLen'], 1, df['SpeedupKernel'], alpha=0.12, color='#9f7aea')

for _, row in df.iterrows():
    ax.annotate(f"{row['SpeedupKernel']:.1f}×",
                xy=(row['SeqLen'], row['SpeedupKernel']),
                xytext=(0, 12), textcoords='offset points',
                color='#9f7aea', fontsize=10, ha='center', fontweight='bold')

ax.set_xlabel('Sequence Length (bp)', fontsize=13)
ax.set_ylabel('Speedup (×)', fontsize=13)
ax.set_title('CUDA GPU Speedup over CPU Sequential\nNeedleman-Wunsch DNA Alignment | BCS702 CCA-1', fontsize=14, fontweight='bold', pad=14)
ax.legend(fontsize=11, facecolor='#21262d', edgecolor='#30363d', labelcolor='#c9d1d9')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('cuda_speedup.png', dpi=150, facecolor=BG)
plt.show()
print('✅ Saved: cuda_speedup.png')

In [ ]:
# ── CELL 6: Print final summary table ──────────────────────
import pandas as pd

df = pd.read_csv('results.csv')
print('\n' + '='*80)
print('  FINAL CUDA BENCHMARK RESULTS — BCS702 CCA-1')
print('  Needleman-Wunsch DNA Alignment | GPU: NVIDIA T4')
print('='*80)
print(f"  {'Size':>6}  {'CPU ms':>10}  {'GPU Kernel':>12}  {'GPU Total':>10}  {'Speedup-K':>10}  {'Status':>6}")
print('-'*80)
for _, r in df.iterrows():
    print(f"  {int(r['SeqLen']):>6}  {r['CPU_ms']:>10.2f}  {r['GPUKernel_ms']:>12.2f}  {r['GPUTotal_ms']:>10.2f}  {r['SpeedupKernel']:>9.2f}x  {r['Validation']:>6}")
print('='*80)
print(f"\n  Max GPU Kernel Speedup: {df['SpeedupKernel'].max():.2f}x  (at {int(df.loc[df['SpeedupKernel'].idxmax(),'SeqLen'])} bp)")
print(f"  All validations PASS  : {'YES ✓' if (df['Validation']=='PASS').all() else 'NO ✗'}")

In [ ]:
# ── CELL 7: Download results ────────────────────────────────
from google.colab import files

files.download('results.csv')
files.download('cuda_time_comparison.png')
files.download('cuda_speedup.png')

print('✅ Downloaded: results.csv, cuda_time_comparison.png, cuda_speedup.png')
print('   Copy results.csv to C:\\DNA-CUDA-Alignment\\results\\cuda_benchmark.csv')
print('   Copy PNG files  to C:\\DNA-CUDA-Alignment\\results\\')